In [0]:
import pyspark.sql.functions as F
from pyspark.sql.types import *

In [0]:
%run ../1_setup/utils

In [0]:
## Enter manually the S3 uri to access data.

dbutils.widgets.text("data_source", "s3 uri", "Data Source")
# e.g. s3://bucket_name/customers/*.csv
data_source = dbutils.widgets.get("data_source")
print(data_source)

## Bronze

In [0]:
df_bronze = spark.read.csv(data_source, header=True, inferSchema=True)
display(df_bronze.limit(5))

In [0]:
df_bronze = df_bronze.withColumn('ingested_at', F.current_timestamp())\
    .withColumn('_source_file', F.col('_metadata.file_path'))\
    .withColumn('_file_size', F.col('_metadata.file_size'))

df_bronze.printSchema()

In [0]:
df_bronze.write.mode('overwrite')\
    .format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{bronze_schema}.dim_products')

## Silver


In [0]:
df_bronze = spark.table(f'{catalog}.{bronze_schema}.dim_products')
display(df_bronze.limit(10))

In [0]:
df_silver = df_bronze.select('product_id', 'product_name', 'category')

In [0]:
print('Rows before duplicates dropped: ', df_bronze.count())
df_silver = df_bronze.dropDuplicates(['product_id'])
print('Rows after duplicates dropped: ', df_silver.count())

In [0]:
df_silver.select('category').distinct().show()

In [0]:
df_silver = df_silver.withColumn('category', F.initcap('category'))
df_silver.select('category').distinct().show()

In [0]:
df_silver = df_silver\
    .withColumn( 'product_name', F.regexp_replace(F.col('product_name'), '(?i)Protien', 'Protein'))\
    .withColumn( 'category', F.regexp_replace(F.col('category'), '(?i)Protien', 'Protein'))
display(df_silver.limit(5))


In [0]:
# Standardizing Customer Attributes to Match Parent Company Data Model

# 1: Add division column
df_gold = (
    df_silver
    .withColumn(
        'division',
        F.when(F.col('category') == 'Energy Bars',        'Nutrition Bars')
         .when(F.col('category') == 'Protein Bars',       'Nutrition Bars')
         .when(F.col('category') == 'Granola & Cereals',  'Breakfast Foods')
         .when(F.col('category') == 'Recovery Dairy',     'Dairy & Recovery')
         .when(F.col('category') == 'Healthy Snacks',     'Healthy Snacks')
         .when(F.col('category') == 'Electrolyte Mix',    'Hydration & Electrolytes')
         .otherwise('Other')
    )
)


# 2: Variant column
df_gold = df_gold.withColumn(
    'variant',
    F.regexp_extract(F.col('product_name'), r'\((.*?)\)', 1)
)


# 3: Create new column: product_code  
# Invalid product_ids are replaced with a fallback value to avoid losing fact records and ensure downstream joins remain consistent

df_gold = (
    df_gold
    # 1. Generate deterministic product_code from product_name
    .withColumn(
        'product_code',
        F.sha2(F.col('product_name').cast('string'), 256)
    )
    # 2. Clean product_id: keep only numeric IDs, else set to 999999
    .withColumn(
        'product_id',
        F.when(
            F.col('product_id').cast('string').rlike('^[0-9]+$'),
            F.col('product_id').cast('string')
        ).otherwise(F.lit(999999).cast('string'))
    )
    # 3. Rename product_name → product
    .withColumnRenamed('product_name', 'product')
)

In [0]:
df_silver.write.mode('overwrite')\
    .format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{silver_schema}.dim_products')

## Gold

In [0]:
df_silver = spark.table(f'{catalog}.{silver_schema}.dim_products')
display(df_silver.limit(10))

In [0]:
df_gold = df_gold.select('product_code', 'division', 'category', 'product', 'variant', 'product_id', 'ingested_at', '_source_file', '_file_size')

In [0]:
display(df_gold)

In [0]:
df_gold.write.mode('overwrite')\
    .format('delta')\
    .option("delta.enableChangeDataFeed", "true")\
    .saveAsTable(f'{catalog}.{gold_schema}.child_dim_products')

## Merging Child Company with Parent

In [0]:

%sql
MERGE INTO fmcg.gold.dim_products AS target
USING fmcg.gold.child_dim_products AS source
ON target.product_code = source.product_code
WHEN MATCHED THEN
  UPDATE SET
    division = source.division,
    category = source.category,
    product = source.product,
    variant = source.variant
WHEN NOT MATCHED THEN
  INSERT (product_code, division, category, product, variant)
  VALUES (source.product_code, source.division, source.category, source.product, source.variant)